# 04 · SKU consolidation

The same product appears under several SKUs and names. SKUs that share any product name are merged into one canonical product (union-find over the sku <-> name graph).

In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = next(d for d in ('data', '../data', '../../data') if os.path.isdir(d))
MISSING_TOKENS = {'', 'null', 'n/a', 'na', 'nan', 'none'}
DECISION_LOG = []

def log_decision(step, decision, rows_touched, rows_before=None, rows_after=None):
    DECISION_LOG.append({'step': step, 'decision': decision, 'rows_touched': int(rows_touched),
                         'rows_before': rows_before, 'rows_after': rows_after})


In [2]:
CATEGORY_MAP = {
    'alcohol': 'Alcohol', 'alcoholic beverages': 'Alcohol',
    'baby': 'Baby', 'baby products': 'Baby',
    'breakfast': 'Breakfast', 'breakfast foods': 'Breakfast',
    'electronics': 'Electronics',
    'emergency': 'Emergency', 'emergency supplies': 'Emergency',
    'entertainment': 'Entertainment', 'games & entertainment': 'Entertainment',
    'grocery': 'Grocery', 'groceries': 'Grocery',
    'health': 'Health', 'health & wellness': 'Health',
    'household': 'Household', 'household essentials': 'Household',
    'snacks': 'Snacks',
}
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%m/%d/%Y %H:%M',
              '%d-%b-%Y %H:%M', '%m-%d-%Y %H:%M:%S']

def load_transactions():
    return pd.read_csv(os.path.join(DATA_DIR, 'walmart_transactions.csv'), dtype=str)

def standardize_missing(df):
    """Strip whitespace; turn '', NULL, N/A, NaN... into real NaN."""
    out = df.copy()
    touched = 0
    for c in out.columns:
        s = out[c].str.strip()
        newly_na = s.str.lower().isin(MISSING_TOKENS) & out[c].notna()
        touched += int(newly_na.sum())
        out[c] = s.mask(newly_na)
    log_decision('missing', 'strip whitespace; blank/NULL/N-A tokens -> NaN (cells, not rows)', touched, len(df), len(out))
    return out

def remove_duplicates(df):
    n0 = len(df)
    out = df.drop_duplicates()
    log_decision('duplicates', 'drop exact duplicate rows (keep first)', n0 - len(out), n0, len(out))
    n1 = len(out)
    out = out.drop_duplicates('transaction_id')
    log_decision('duplicates', 'drop repeated transaction_id (keep first)', n1 - len(out), n1, len(out))
    return out

def clean_unit_price(df):
    out = df.copy()
    raw = out['unit_price']
    dollar = int(raw.str.contains('$', regex=False, na=False).sum())
    out['unit_price'] = pd.to_numeric(raw.str.replace('$', '', regex=False).str.replace(',', '', regex=False), errors='coerce')
    bad = int(out['unit_price'].isna().sum() - raw.isna().sum())
    log_decision('unit_price', "remove '$', cast text -> float", dollar, len(df), len(out))
    log_decision('unit_price', 'values that still failed numeric cast -> NaN', bad, len(df), len(out))
    return out

def clean_quantity(df):
    out = df.copy()
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    n0 = len(out)
    out = out[out['quantity'] > 0]
    log_decision('quantity', 'drop rows with negative/zero/unparseable quantity (entry errors; sign-flip not assumed)', n0 - len(out), n0, len(out))
    return out

def parse_timestamps(df):
    out = df.copy()
    ts = pd.Series(pd.NaT, index=out.index, dtype='datetime64[ns]')
    for fmt in TS_FORMATS:
        todo = ts.isna() & out['timestamp'].notna()
        ts[todo] = pd.to_datetime(out.loc[todo, 'timestamp'], format=fmt, errors='coerce')
    unparsed = int((ts.isna() & out['timestamp'].notna()).sum())
    missing = int(out['timestamp'].isna().sum())
    out['timestamp'] = ts
    log_decision('timestamp', f'parse {len(TS_FORMATS)} mixed formats into one datetime (naive local time); {unparsed} unparseable', int(ts.notna().sum()), len(df), len(out))
    log_decision('timestamp', 'missing timestamps kept (NaT) but excluded from any date-based analysis; not imputed', missing, len(df), len(out))
    return out

def clean_store_id(df):
    out = df.copy()
    raw = out['store_id']
    out['store_id'] = (raw.str.upper().str.replace('_', '-', regex=False)
                          .str.replace(r'^FL-?(\d{3})$', r'FL-\1', regex=True))
    log_decision('store_id', "uppercase, strip, '_'/missing '-' -> canonical 'FL-###'", int((raw != out['store_id']).sum()), len(df), len(out))
    return out

def clean_category(df):
    out = df.copy()
    raw = out['category']
    out['category'] = raw.str.lower().map(CATEGORY_MAP)
    log_decision('category', f'collapse {raw.nunique()} spellings into {out["category"].nunique()} canonical categories', int((raw != out['category']).sum()), len(df), len(out))
    return out

def clean_transactions():
    DECISION_LOG.clear()
    df = load_transactions()
    for step in (standardize_missing, remove_duplicates, clean_unit_price, clean_quantity,
                 parse_timestamps, clean_store_id, clean_category):
        df = step(df)
    df['product_name_raw'] = df['product_name']
    return df.reset_index(drop=True)


In [3]:
def build_sku_map(df):
    """Group SKUs that share product names (union-find over sku <-> normalized name)."""
    names = df[['sku', 'product_name']].dropna().assign(name=lambda d: d['product_name'].str.lower().str.strip())
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for sku, name in names[['sku', 'name']].drop_duplicates().itertuples(index=False):
        parent[find(('sku', sku))] = find(('name', name))
    names['group'] = names['sku'].map(lambda s: find(('sku', s)))
    sku_group = names.drop_duplicates('sku').set_index('sku')['group']
    ids = {g: i + 1 for i, g in enumerate(sorted(sku_group.unique(), key=str))}
    # canonical name = most common mixed-case product_name in the group (skips ALL-CAPS / lowercase variants)
    mixed = names[~names['product_name'].str.isupper() & ~names['product_name'].str.islower()]
    top_name = (mixed.assign(pid=mixed['group'].map(ids)).groupby('pid')['product_name']
                .agg(lambda s: s.str.strip().value_counts().idxmax()))
    out = pd.DataFrame({'sku': sku_group.index, 'canonical_id': sku_group.map(ids).values})
    out['canonical_name'] = out['canonical_id'].map(top_name)
    return out.sort_values(['canonical_id', 'sku']).reset_index(drop=True)


In [4]:
tx_clean = clean_transactions()
sku_map = build_sku_map(tx_clean)
print('distinct SKUs before:', sku_map['sku'].nunique())
print('canonical products after:', sku_map['canonical_id'].nunique())
sizes = sku_map.groupby(['canonical_id', 'canonical_name']).size().sort_values(ascending=False)
print('largest consolidated group:', sizes.index[0][1], f'({sizes.iloc[0]} SKUs)')
sku_map

distinct SKUs before: 45
canonical products after: 20
largest consolidated group: Pop-Tarts Strawberry (4 SKUs)


,sku,canonical_id,canonical_name
0,DIAP-S3,1,Diapers Size 3
1,DIAPERS-3,1,Diapers Size 3
2,BABYFORMULA,2,Infant Formula
3,BFORM-01,2,Infant Formula
4,BAT-AA4,3,AA Batteries 4-Pack
5,BATT-100,3,AA Batteries 4-Pack
6,BATT_AA_4PK,3,AA Batteries 4-Pack
7,BEER-24,4,Light Beer 24pk
8,LTBEER24,4,Light Beer 24pk
9,WATER24PK,5,Bottled Water 24-Pack


In [5]:
# Pop-Tarts family: every SKU and every spelling that now maps to one product
pt_id = sku_map.loc[sku_map['canonical_name'].str.contains('Pop', case=False), 'canonical_id'].iloc[0]
print(sku_map[sku_map['canonical_id'] == pt_id])
pt_names = tx_clean.loc[tx_clean['sku'].isin(sku_map.loc[sku_map['canonical_id'] == pt_id, 'sku']), 'product_name_raw']
print('\nraw name spellings:', pt_names.nunique()); print(pt_names.value_counts())

              sku  canonical_id        canonical_name
33  POPTART-STRAW            16  Pop-Tarts Strawberry
34          PT-12            16  Pop-Tarts Strawberry
35  PT_STRAWBERRY            16  Pop-Tarts Strawberry
36     SKU#459812            16  Pop-Tarts Strawberry

raw name spellings: 5
product_name_raw
Pop-Tarts Strawberry      668
POPTART-STRAWBERRY        383
Pop Tarts - Strawberry    356
pop-tarts strawberry      327
Strawberry Pop-Tart       319
Name: count, dtype: int64


In [6]:
# Cross-check against the product catalog: SKUs we merged should have matching catalog names
catalog = pd.read_csv(os.path.join(DATA_DIR, 'product_catalog (1).csv'))
chk = sku_map.merge(catalog[['sku', 'product_name']].rename(columns={'product_name': 'catalog_name'}), on='sku', how='outer', indicator=True)
print(chk['_merge'].value_counts())
chk[chk['canonical_id'].isin([pt_id])]

_merge
both          45
left_only      0
right_only     0
Name: count, dtype: int64


,sku,canonical_id,canonical_name,catalog_name,_merge
29,POPTART-STRAW,16,Pop-Tarts Strawberry,Pop-Tarts Strawberry,both
31,PT-12,16,Pop-Tarts Strawberry,pop-tarts strawberry,both
33,PT_STRAWBERRY,16,Pop-Tarts Strawberry,pop-tarts strawberry,both
35,SKU#459812,16,Pop-Tarts Strawberry,Strawberry Pop-Tart,both


In [7]:
# Before vs after, daily Pop-Tart units (preview of Q2)
tx_clean = tx_clean.merge(sku_map, on='sku', how='left')
assert tx_clean['canonical_id'].notna().all()
d = tx_clean.dropna(subset=['timestamp']).assign(day=lambda x: x['timestamp'].dt.date)
pt = d[d['canonical_id'] == pt_id]
print('largest single SKU, avg daily units :', pt.groupby(['day', 'sku'])['quantity'].sum().groupby('sku').mean().max().round(1))
print('consolidated product, avg daily units:', pt.groupby('day')['quantity'].sum().mean().round(1))

largest single SKU, avg daily units : 44.4
consolidated product, avg daily units: 166.8
